In [1]:
load('sim.sage')

In [2]:
class TopKCollector:
    def __init__(self, specials=None, multiplier=2, max_cache=200):
        self.specials = specials
        self.multiplier = multiplier
        self.max_cache = max_cache
        self._seen = {}  # latex_str -> (energy, expr)
        self.chosen = None

    def accept(self, state, E):
        expr = state[0]
        lx = str(latex(expr))
        if lx not in self._seen or E < self._seen[lx][0]:
            self._seen[lx] = (E, expr)
        if len(self._seen) > 2 * self.max_cache:
            ranked = sorted(self._seen.items(), key=lambda kv: kv[1][0])
            self._seen = dict(ranked[:self.max_cache])

    def results(self, K=10):
        ranked = sorted(self._seen.values(), key=lambda t: t[0])[:K]
        return [(i+1, e, expr) for i, (e, expr) in enumerate(ranked)]


_last_collector = [None]

def SimTopK(ex, K=10, Tmax=100, Tmin=1, steps=100,
            _TL=TL_default, specials=None, multiplier=2):
    collector = TopKCollector(specials=specials, multiplier=multiplier)
    Sim(ex, Tmax=Tmax, Tmin=Tmin, steps=steps,
        _TL=_TL, specials=specials, multiplier=multiplier,
        on_accept=collector.accept)
    _last_collector[0] = collector
    return collector

In [3]:
from IPython.display import display, HTML, Latex
import ipywidgets as widgets

def show_candidates(collector, expr=None, K=3):
    results = collector.results(K)
    if not results:
        display(HTML('<p><b>No simplifications found.</b></p>'))
        return

    # Show original expression
    if expr is not None:
        lx_orig = str(latex(expr))
        orig_len = lenl(expr, collector.specials, collector.multiplier)
        display(HTML(
            '<h3>Original (length %d)</h3>'
            '<p style="font-size:1.3em">$\\displaystyle %s$</p><hr>' % (int(orig_len), lx_orig)))

    n_total = len(collector._seen)
    display(HTML('<h3>Top %d Simplifications (%d unique found)</h3>' % (len(results), n_total)))

    # Output area for showing the selected expression
    selection_output = widgets.Output()

    def make_click_handler(rank, expression):
        def on_click(b):
            collector.chosen = expression
            for btn in buttons:
                btn.style.button_color = None
            b.style.button_color = '#d0f0d0'
            with selection_output:
                selection_output.clear_output()
                display(HTML('<b>Selected #%d</b>' % rank))
                show(expression)
        return on_click

    buttons = []
    for rank, energy, expression in results:
        lx = str(latex(expression))
        btn = widgets.Button(
            description='#%d (len %d)' % (rank, int(energy)),
            layout=widgets.Layout(width='auto', min_width='100px'))
        btn.on_click(make_click_handler(rank, expression))
        buttons.append(btn)
        # Use Output widget so HTML/MathJax renders properly
        label_out = widgets.Output()
        with label_out:
            display(HTML(
                '<span style="font-size:1.2em;vertical-align:middle">'
                '$\\displaystyle %s$</span>' % lx))
        display(widgets.HBox([btn, label_out],
            layout=widgets.Layout(align_items='center', margin='4px 0')))

    display(HTML('<hr>'))
    display(selection_output)


def pick(n, collector=None):
    if collector is None:
        collector = _last_collector[0]
    if collector is None:
        raise ValueError('No collector - run SimTopK first')
    results = collector.results()
    if n < 1 or n > len(results):
        raise ValueError('Pick between 1 and %d' % len(results))
    expr = results[n-1][2]
    collector.chosen = expr
    show(expr)
    return expr


def merge(c1, c2):
    if (c1.specials, c1.multiplier) != (c2.specials, c2.multiplier):
        raise ValueError(
            'Cannot merge collectors with different specials/multiplier - '
            'their energy values are not comparable')
    merged = TopKCollector(specials=c1.specials, multiplier=c1.multiplier)
    merged._seen = dict(c1._seen)
    for lx, (e, expr) in c2._seen.items():
        if lx not in merged._seen or e < merged._seen[lx][0]:
            merged._seen[lx] = (e, expr)
    return merged

In [4]:
var('a b c x')
expr = sqrt((a^3 - b^3)/(a - b) + a*b)

c = SimTopK(expr, steps=500, _TL=TL_sim)
show_candidates(c, expr)

Output()

In [5]:
# After clicking a button above, the selection is stored in c.chosen
# You can also pick by number: result = pick(2)
result = c.chosen
result

In [6]:
# Re-run with more steps and/or different transformations, then merge
c2 = SimTopK(expr, steps=1000, _TL=TL_ALL)
c_all = merge(c, c2)
show_candidates(c_all, expr)

Output()

---
## More Examples

In [7]:
# From sim.sage docs: an integral that produces a messy closed form
var('a b c x')
ex = integral(1/((a*x+b)^2*(c*x+c)^2), x)
c1 = SimTopK(ex, steps=1000, _TL=TL_sim)
show_candidates(c1, ex, K=5)

Output()

In [8]:
# Trig expression with many equivalent forms
var('t')
ex = (sin(t)^2 + cos(t)^2)^2 + sin(2*t) - 2*sin(t)*cos(t)
c2 = SimTopK(ex, steps=500, _TL=TL_sim + TL_trig)
show_candidates(c2, ex, K=5)

Output()

In [9]:
# Nested radicals
var('a b')
ex = sqrt(a + sqrt(a^2 - b^2)) + sqrt(a - sqrt(a^2 - b^2))
c3 = SimTopK(ex, steps=800, _TL=TL_sim)
show_candidates(c3, ex, K=5)

Output()

In [10]:
# Logarithmic identity buried in noise
var('x y')
ex = log(x^2*y) - 2*log(x) + log(exp(x)*exp(y)) - x
c4 = SimTopK(ex, steps=500, _TL=TL_sim + TL_log)
show_candidates(c4, ex, K=5)

Output()

---
## Beta Distribution: Simplification with `discover_substitutions()`

This reproduces the paper's main result (Sim.pdf): simplifying the peak of the beta distribution's triangular envelope from **564 characters to 66**, now using automatic substitution discovery instead of manual dictionary construction.

In [ ]:
# Load betaT.sage which computes xp (the triangular envelope peak)
load('betaT.sage')
print("Original xp has LaTeX length:", lenl(xp))
show(xp)

In [ ]:
# Multi-run annealing with auto-discovered substitutions
# Each run: anneal -> discover subs -> iterate with subs -> inversion trick -> re-anneal
best = xp
best_len = lenl(xp)
NUM_RUNS = 5

for run in range(NUM_RUNS):
    # First pass
    x1 = Sim(xp, steps=3000)
    dic1 = discover_substitutions(x1)
    
    # Iterative passes with substitutions
    prev = x1
    for i in range(5):
        prev = Sim(prev, subs_dic=dic1, steps=2000)
    
    # Inversion trick + re-anneal with subs
    ix = Sim(1/prev, steps=3000, subs_dic=dic1)
    for i in range(3):
        ix = Sim(ix, subs_dic=dic1, steps=2000)
    
    result = 1/ix
    l = lenl(result)
    print(f"Run {run+1}/{NUM_RUNS}: length {l}")
    if l < best_len:
        best = result
        best_len = l
        best_dic = dic1
        print(f"  NEW BEST!")

print(f"\n{'='*50}")
print(f"Original length: {lenl(xp)}")
print(f"Final length:    {best_len}")
print(f"Reduction:       {lenl(xp)/best_len:.1f}x shorter")
print(f"Paper target:    66")

print(f"\nDiscovered substitutions:")
for k, v in sorted(best_dic.items(), key=lambda t: str(t[0])):
    if str(v)[0] != '-':
        print(f"  {k}  ->  {v}")

print(f"\nSimplified form:")
show(best)